In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS capgeminipro.retail_gold;

In [0]:
%sql
CREATE OR REPLACE TABLE capgeminipro.retail_silver.silver_customers_delta

USING DELTA

AS

SELECT *

FROM capgeminipro.retail_silver.silver_customers;

In [0]:
%sql
DESCRIBE DETAIL capgeminipro.retail_silver.silver_customers_delta;

In [0]:
%sql
CREATE OR REPLACE TABLE capgeminipro.retail_silver.silver_products_delta

USING DELTA

AS

SELECT *

FROM capgeminipro.retail_silver.silver_products_clean;

In [0]:
%sql
CREATE OR REPLACE TABLE capgeminipro.retail_silver.silver_stores_delta

USING DELTA

AS

SELECT *

FROM capgeminipro.retail_silver.silver_stores_clean;

In [0]:
%sql
CREATE OR REPLACE TABLE capgeminipro.retail_silver.silver_sales_delta

USING DELTA

AS

SELECT *

FROM capgeminipro.retail_silver.silver_sales_final;

In [0]:
%sql
CREATE OR REPLACE TABLE capgeminipro.retail_gold.dim_customers (

    CustomerSK BIGINT GENERATED ALWAYS AS IDENTITY,

    CustomerID INT,

    CustomerName STRING,

    Email STRING,

    City STRING,

    Address STRING,

    StartDate DATE,

    EndDate DATE,

    IsCurrent STRING

)

USING DELTA;

In [0]:
%sql
INSERT INTO capgeminipro.retail_gold.dim_customers (

    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsCurrent

)

SELECT

    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    LastUpdated,
    NULL,
    'Y'

FROM capgeminipro.retail_silver.silver_customers_delta;

In [0]:
%sql
SELECT *
FROM capgeminipro.retail_gold.dim_customers
ORDER BY CustomerID;

In [0]:
%sql
CREATE OR REPLACE TABLE capgeminipro.retail_gold.dim_products (

    ProductSK BIGINT GENERATED ALWAYS AS IDENTITY,

    ProductID INT,

    ProductName STRING,

    Category STRING,

    UnitPrice DECIMAL(10,2)

)

USING DELTA;

In [0]:
%sql
INSERT INTO capgeminipro.retail_gold.dim_products (

    ProductID,
    ProductName,
    Category,
    UnitPrice

)

SELECT

    ProductID,
    ProductName,
    Category,
    UnitPrice

FROM capgeminipro.retail_silver.silver_products_delta;

In [0]:
%sql
SELECT *
FROM capgeminipro.retail_gold.dim_products;

In [0]:
%sql

CREATE OR REPLACE TABLE capgeminipro.retail_gold.dim_stores (

    StoreSK BIGINT GENERATED ALWAYS AS IDENTITY,

    StoreID INT,

    StoreName STRING,

    Region STRING

)

USING DELTA;

In [0]:
%sql

INSERT INTO capgeminipro.retail_gold.dim_stores (

    StoreID,
    StoreName,
    Region

)

SELECT

    StoreID,
    StoreName,
    Region

FROM capgeminipro.retail_silver.silver_stores_delta;

In [0]:
%sql

SELECT *

FROM capgeminipro.retail_gold.dim_stores;

In [0]:
%sql

CREATE OR REPLACE TABLE capgeminipro.retail_gold.fact_sales (

    SalesSK BIGINT GENERATED ALWAYS AS IDENTITY,

    TransactionID INT,

    CustomerSK BIGINT,

    ProductSK BIGINT,

    StoreSK BIGINT,

    Quantity INT,

    TxnDate DATE

)

USING DELTA;

In [0]:
%sql

INSERT INTO capgeminipro.retail_gold.fact_sales (

    TransactionID,
    CustomerSK,
    ProductSK,
    StoreSK,
    Quantity,
    TxnDate

)

SELECT

    s.TransactionID,

    dc.CustomerSK,

    dp.ProductSK,

    ds.StoreSK,

    s.Quantity,

    s.TxnDate

FROM capgeminipro.retail_silver.silver_sales_delta s

INNER JOIN capgeminipro.retail_gold.dim_customers dc
ON s.CustomerID = dc.CustomerID
AND dc.IsCurrent = 'Y'

INNER JOIN capgeminipro.retail_gold.dim_products dp
ON s.ProductID = dp.ProductID

INNER JOIN capgeminipro.retail_gold.dim_stores ds
ON s.StoreID = ds.StoreID;

In [0]:
%sql

SELECT *

FROM capgeminipro.retail_gold.fact_sales;

In [0]:
%sql

CREATE OR REPLACE TABLE capgeminipro.retail_gold.fact_sales AS

SELECT

    monotonically_increasing_id() AS SalesSK,

    s.TransactionID,

    dc.CustomerSK,

    dp.ProductSK,

    ds.StoreSK,

    s.Quantity,

    (s.Quantity * dp.UnitPrice) AS Amount,

    s.TxnDate

FROM capgeminipro.retail_silver.silver_sales_delta s

INNER JOIN capgeminipro.retail_gold.dim_customers dc
ON s.CustomerID = dc.CustomerID
AND dc.IsCurrent = 'Y'

INNER JOIN capgeminipro.retail_gold.dim_products dp
ON s.ProductID = dp.ProductID

INNER JOIN capgeminipro.retail_gold.dim_stores ds
ON s.StoreID = ds.StoreID;